# ボール打撃だけをACTで学習
1. ランタイムのタイプを変更し、GPUを選択。
2. ball_strike_colab.zipをアップロード。
3. セルを上から実行。教師データの再現を学習するもので、的に当たる性能を保証しません。
依存関係のエラーがあればそこで止めて確認してください。ライブラリ更新後に再起動を求められた場合はランタイムを再起動します。

In [ ]:
from google.colab import files
uploaded = files.upload()
from pathlib import Path
import zipfile
with zipfile.ZipFile('ball_strike_colab.zip') as z:
    z.extractall('/content')
assert Path('/content/submission_ball_strike/data/strike_lerobot/experiment.json').exists()


In [ ]:
import subprocess, sys
p=Path('/content/lerobot_pinned')
if not p.exists():
    subprocess.run(['git','clone','https://github.com/huggingface/lerobot.git',str(p)],check=True)
subprocess.run(['git','-C',str(p),'checkout','768e36660d1408c71118a2760f831c037fbfa17d'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-c','/content/submission_ball_strike/constraints-colab.txt','-e',str(p)],check=True)


In [ ]:
import torch, json, subprocess, sys
assert torch.cuda.is_available(), 'GPUランタイムを選択してください'
print(torch.__version__, torch.cuda.get_device_name(0))
cfg=json.loads(Path('/content/submission_ball_strike/configs/learning.json').read_text())
print('Training steps:',cfg['training_steps'])
subprocess.run([sys.executable,'/content/submission_ball_strike/scripts/strike_learning.py','train','--device','cuda','--run-name','act_strike'],check=True)


In [ ]:
from pathlib import Path
import csv
import matplotlib.pyplot as plt
model=Path('/content/submission_ball_strike/checkpoints/act_strike')
assert (model/'model.safetensors').exists()
with (model/'loss.csv').open() as f: rows=list(csv.DictReader(f))
plt.plot([int(r['step']) for r in rows],[float(r['loss']) for r in rows])
plt.xlabel('training step');plt.ylabel('training loss');plt.show()
print('学習損失であり、実際の打撃成功率ではありません。')


In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/act_strike','zip',str(model))
files.download('/content/act_strike.zip')
print('ZIPの中身をWindowsの submission_ball_strike/checkpoints/act_strike/ に展開してください。')
